In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。
```python 
from langchain.chat_models import init_chat_model
model = init_chat_model(model="gpt-5.2")#基于名称推断你要引入的大模型
```

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`/`uv add langchain`
- 在.env中配置模型的api_key
    - apikey这个名字要在Langchain官网确定名字
- 调用init_chat_model函数，传入正确的模型名称
    - 就是前面自动推断的部分

In [ ]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
# 仅限于官方文档中支持的模型
model = init_chat_model(model="deepseek-chat")

In [ ]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

### 自定义模型提供商

init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [ ]:
# 我们收到加载环境变量中的base_url和api_key
import os

#首先是从我们的环境变量中读取我们需要的URL和API_KEY
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max（阿里云使用的模型）
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,#自己指定URL
    api_key=api_key
)
#访问的实际是我们自己指定的URL，使用的是我们自己的API_KEY

In [ ]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [ ]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5
    #配置参数的鹅地方都是在这里的
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


## 1.2.使用model类
不是默认提供的模型（ds）同时也不兼容openai类似格式访问的时候：使用传统社区方法来调用这个模型。

其实init_chat_model函数底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi#连接的通义的apikey

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [ ]:
# 打印结果
print(type(model))

# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [ ]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果。这里就是直接调用的结果
response = model.invoke("你是谁？")#为了方便直接输入访问信息

In [ ]:
# 查看响应内容
print(response)

In [ ]:
# 调用invoke函数，传入消息数组
# 相较于我们之前那配置的msg中药包含LLM的选择，这里由于前面确定使用函数的时候已经完成了模型的选择，所以我们在消息中只需要包含角色和内容即可
response = model.invoke([
    {
        "role": "system", 
        "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"
    },
    {
        "role": "user",
        "content": "你是谁？"
    }
])
print(response.content)


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [ ]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [ ]:
# 打印stream类型
print(type(stream))#类型是生成器，也就是说不是一次性生成所有的结果，而是边生成边返回结果

In [ ]:
for chunk in stream: #取出其片段
    print(chunk.content, end="", flush=True)# 每一次都是返回一个token的内容
    #end=""表示不换行，flush=True表示立即输出结果，不进行缓冲

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个**create_agent函数**用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [ ]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)#这里的model其实就是最开始我使用init定义的初始化模型的设置

In [ ]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")#指定模型名称，由LangChain自动初始化模型

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict（对象），其中必须包含一个messages字段，也就是消息的列表。

主要就是相关的格式不是一样的；

1. 使用model时：
```python
response = model.invoke(
    [{"role": "user", "content": "你是谁？"}]
)
```
2. 使用agent时：
```python
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})
```
主要的区别是：使用model的时候，model相关参数的配置在创建这个model的时候就已经配置完成了，我们这里仅仅添加了msg中需要添加的内容；
在agent中我们传入的信息是一个对象，里面对应的其实是一个数组，具体使用见后面

### 阻塞式调用

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

### 流式访问


In [ ]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"#决定agent输出的时候是以着怎样的模式进行输出的
)
print(type( messages))

In [ ]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content 是否有数据
        print(token.content, end="", flush=True)  # Print token